In [44]:
# Import libraries
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, KNNBasic, SVD, accuracy
from surprise.model_selection import train_test_split, cross_validate
from surprise import AlgoBase
import numpy as np

In [45]:
ratings = pd.read_csv('movies.csv')
ratings.head()

,movieId,title,overview,genre,director,userId,rating,timestamp
0,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,23,3.5,1148721092
1,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,102,4.0,956598942
2,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,232,2.0,955092697
3,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,242,5.0,956688825
4,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,263,3.0,1117846575


In [46]:
movie_features = pd.read_csv('movies.csv')
movie_features = movie_features.dropna()
movie_features.isnull().sum()

movieId      0
title        0
overview     0
genre        0
director     0
userId       0
rating       0
timestamp    0
dtype: int64

In [47]:
movie_features.info()

<class 'pandas.DataFrame'>
Index: 44623 entries, 0 to 45005
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   movieId    44623 non-null  int64  
 1   title      44623 non-null  str    
 2   overview   44623 non-null  str    
 3   genre      44623 non-null  str    
 4   director   44623 non-null  str    
 5   userId     44623 non-null  int64  
 6   rating     44623 non-null  float64
 7   timestamp  44623 non-null  int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 18.0 MB


In [48]:
# Convert the data into a format Surprise can use
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
# Create a training and testing set using surprise
trainset, testset = train_test_split(data, test_size=0.25)

In [49]:
# User-based collaborative filtering
user_based = KNNBasic(k=50, sim_options={'name': 'pearson', 'user_based': True})
user_based.fit(trainset)
# Make predictions on the test set
user_based_predictions = user_based.test(testset)
# Evaluate the performance
user_accuracy = accuracy.rmse(user_based_predictions)
print(f"User-based CF RMSE: {user_accuracy}")

Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 0.9884
User-based CF RMSE: 0.9883790232825218


In [50]:
# Item-based collaborative filtering
item_based = KNNBasic(k=50, sim_options={'name': 'pearson', 'user_based': False})
item_based.fit(trainset)
# Make predictions on the test set
item_based_predictions = item_based.test(testset)
# Evaluate the performance
item_accuracy = accuracy.rmse(item_based_predictions)
print(f"Item-based CF RMSE: {item_accuracy}")

Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 0.9822
Item-based CF RMSE: 0.9822399690682442


In [51]:
# Matrix Factorization using SVD
svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02)
svd_model.fit(trainset)
# Make predictions on the test set
svd_predictions = svd_model.test(testset)
# Evaluate the performance
svd_accuracy = accuracy.rmse(svd_predictions)
print(f"Matrix Factorization RMSE: {svd_accuracy}")

RMSE: 0.8921
Matrix Factorization RMSE: 0.8920989761553563


In [52]:
class ContentBasedAlgorithm(AlgoBase):
    def __init__(self, movie_features_df):
        AlgoBase.__init__(self)
        self.movie_features = {}
        
        # Define the 9 target genres explicitly inside the class
        target_genres = ['Action', 'Adventure', 'Comedy', 'Drama', 'Fantasy', 
                         'Horror', 'Romance', 'Science Fiction', 'Thriller','Mystery', 'Animation',
                         'Documentary','Music', 'Family', 'War','Western','Crime','Tv Movie','History','Foreign']
        
        # Convert movie features to a dictionary for faster lookup
        for _, row in movie_features_df.iterrows():
            movieId = row['movieId']
            
            # CHOSEN CHANGE: Read the single text value and build the 1/0 array dynamically
            current_genre = str(row['genre']).replace('-', '_') # Handles 'Sci-Fi' -> 'Sci_fi'
            
            features = np.array([1 if genre == current_genre else 0 for genre in target_genres])
            
            self.movie_features[movieId] = features
    
    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        
        # Create user profiles based on their rated movies
        self.user_profiles = {}
        for u in trainset.all_users():
            # Get all movies this user has rated
            items_rated = [j for (j, _) in trainset.ur[u]]
            
            if not items_rated:
                # Handle users with no ratings
                self.user_profiles[u] = np.zeros(9)  # 9 genre features
                continue
                
            # Average the features of all movies this user has rated
            features = [self.movie_features.get(self.trainset.to_raw_iid(j), np.zeros(9)) 
                       for j in items_rated if self.trainset.to_raw_iid(j) in self.movie_features]
            
            if features:
                self.user_profiles[u] = np.mean(features, axis=0)
            else:
                self.user_profiles[u] = np.zeros(9)
        
        return self
    
    def estimate(self, u, i):
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            return self.trainset.global_mean
        
        # Get the raw movie id
        raw_movie_id = self.trainset.to_raw_iid(i)
        
        # If we don't have features for this movie, return global mean
        if raw_movie_id not in self.movie_features:
            return self.trainset.global_mean
        
        # Calculate cosine similarity between user profile and movie features
        user_vector = self.user_profiles[u]
        movie_vector = self.movie_features[raw_movie_id]
        
        # Avoid division by zero
        if np.all(user_vector == 0) or np.all(movie_vector == 0):
            return self.trainset.global_mean
        
        cos_sim = np.dot(user_vector, movie_vector) / (np.linalg.norm(user_vector) * np.linalg.norm(movie_vector))
        
        # Convert similarity to rating scale (1-5)
        # Map similarity from [-1, 1] to [1, 5]
        predicted_rating = 1 + 2 * (cos_sim + 1)
        
        return predicted_rating

# CHOSEN CHANGE: Pass your main DataFrame directly into the class instance
cb_algo = ContentBasedAlgorithm(movie_features)
cb_algo.fit(trainset)

# Make predictions
cb_predictions = cb_algo.test(testset)

# Evaluate
cb_accuracy = accuracy.rmse(cb_predictions)
print(f"Content-based filtering RMSE: {cb_accuracy}")


RMSE: 1.2153
Content-based filtering RMSE: 1.2152751316810257


In [53]:
# Simple weighted hybrid: Combine SVD and content-based predictions
class HybridRecommender:
    def __init__(self, cf_algo, cb_algo, cf_weight=0.7):
        self.cf_algo = cf_algo  # Collaborative filtering algorithm
        self.cb_algo = cb_algo  # Content-based algorithm
        self.cf_weight = cf_weight
    
    def predict(self, user_id, movie_id):
        # Get predictions from both algorithms
        try:
            cf_pred = self.cf_algo.predict(user_id, movie_id).est
        except:
            cf_pred = 3.0  # Default if prediction fails
        
        try:
            cb_pred = self.cb_algo.predict(user_id, movie_id).est
        except:
            cb_pred = 3.0  # Default if prediction fails
        
        # Combine predictions with weighted average
        hybrid_pred = (self.cf_weight * cf_pred) + ((1 - self.cf_weight) * cb_pred)
        
        return hybrid_pred

# Create the hybrid recommender
hybrid = HybridRecommender(svd_model, cb_algo, cf_weight=0.7)

# Evaluate on testset
hybrid_predictions = []
for uid, iid, true_r in testset:
    pred = hybrid.predict(uid, iid)
    hybrid_predictions.append((uid, iid, true_r, pred))

# Calculate RMSE manually
def calculate_rmse(predictions):
    sum_squared_diff = 0
    for _, _, true_r, pred_r in predictions:
        sum_squared_diff += (true_r - pred_r) ** 2
    return np.sqrt(sum_squared_diff / len(predictions))

hybrid_rmse = calculate_rmse(hybrid_predictions)
print(f"Hybrid recommender RMSE: {hybrid_rmse}")

Hybrid recommender RMSE: 0.9212329543543851


In [57]:
# Function to get top N recommendations for a user
def get_top_n_recommendations(userId, n=10, algorithm=svd_model, movies_df=None):
    """
    Generate top N movie recommendations for a specific user
    
    Parameters:
    user_id: The user ID
    n: Number of recommendations
    algorithm: Trained recommendation algorithm
    movies_df: DataFrame with movie information
    
    Returns:
    List of (movie_id, predicted_rating) tuples
    """
    # Get a list of all movies
    all_movie_ids = ratings['movieId'].unique()
    
    # Get movies this user has already rated
    user_ratings = ratings[ratings['userId'] == userId]['movieId'].unique()
    
    # Movies the user hasn't rated
    movies_to_predict = np.setdiff1d(all_movie_ids, user_ratings)
    
    # Predict ratings for all unrated movies
    predictions = [algorithm.predict(userId, movieId) for movieId in movies_to_predict]
    
    # Sort predictions by estimated rating
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # Take top N
    top_n = predictions[:n]
    
    if movies_df is not None:
        # Return movie information along with predictions
        result = []
        for pred in top_n:
            movie_info = movies_df[movies_df['movieId'] == pred.iid]
            if not movie_info.empty:
                result.append((movie_info['title'].values[0], pred.est))
        return result
    else:
        # Just return movie IDs and predicted ratings
        return [(pred.iid, pred.est) for pred in top_n]

# Load movie information
movies = pd.read_csv('movies.csv')
# Get recommendations for a specific user
user_input = input("Enter the User ID (e.g., 50): ")

try:
    # 2. Convert the input to an integer
    target_user_id = int(user_input)
    
    # 3. Get recommendations for the entered user ID
    user_recommendations = get_top_n_recommendations(
        userId=target_user_id, n=5, algorithm=svd_model, movies_df=movies
    )

    # 4. Display the results
    print(f"\nTop 5 recommendations for user {target_user_id}:")
    for title, est in user_recommendations:
        print(f" {title} (predicted rating: {est:.2f})")

except ValueError:
    print("Error: Please enter a valid numerical User ID.")
except Exception as e:
    print(f"An error occurred: {e}. Please ensure the User ID exists in your dataset.")
# user_recommendations = get_top_n_recommendations(
#     userId=50, n=5, algorithm=svd_model, movies_df=movies)

# print("Top 5 recommendations for user 50:")
# for title, est in user_recommendations:
#     print(f"{title} (predicted rating: {est:.2f})")


Top 5 recommendations for user 1000:
 The Million Dollar Hotel (predicted rating: 4.50)
 Sleepless in Seattle (predicted rating: 4.47)
 Galaxy Quest (predicted rating: 4.41)
 The Good Thief (predicted rating: 4.37)
 The Sixth Sense (predicted rating: 4.33)
